# 13. 文件读取、写入和 OS 综合

文件读写用于让程序和外部文件交换数据；`os`、`pathlib` 用于处理路径、目录、文件状态等系统相关操作。

本章把文件读写和 OS 操作放在一起学习，重点内容：

- 路径准备：相对路径、绝对路径、`Path`
- 文本文件读取：`read()`、`readline()`、`readlines()`、逐行遍历
- 文本文件写入：`write()`、`writelines()`、追加模式
- `with open(...)` 自动管理文件关闭
- CSV 文件读取和写入
- JSON 文件读取和写入
- `os` 与 `pathlib` 的常见文件系统操作
- 创建目录、遍历目录、重命名、删除文件
- 常见错误和安全建议

本章每类常见用法都配有示例代码和小练习。


## 1. 路径准备

文件操作第一步是找到文件。初学时最容易出错的不是读写语法，而是路径写错。

建议优先使用 `pathlib.Path`，它比手写字符串路径更清晰，也更适合跨平台。


In [1]:
from pathlib import Path

# 当前 Notebook 通常在 python_learn 目录中运行，此时 res 目录就在当前目录下。
# 如果从项目根目录运行，res 目录路径会变成 python_learn/res。
RES_DIR = Path('res')
if not RES_DIR.exists() and Path('python_learn/res').exists():
    RES_DIR = Path('python_learn/res')

# 创建一个专门给本章写文件用的工作目录，避免影响已有文件。
WORK_DIR = RES_DIR / 'chapter13_workspace'
WORK_DIR.mkdir(exist_ok=True)

text_file = RES_DIR / 'chapter13_text.txt'
csv_file = RES_DIR / 'chapter13_students.csv'
json_file = RES_DIR / 'chapter13_config.json'

print('资源目录：', RES_DIR.resolve())
print('工作目录：', WORK_DIR.resolve())
print('文本文件是否存在：', text_file.exists())
print('CSV 文件是否存在：', csv_file.exists())
print('JSON 文件是否存在：', json_file.exists())


资源目录： C:\Users\11435\Desktop\python_base\python_learn\res
工作目录： C:\Users\11435\Desktop\python_base\python_learn\res\chapter13_workspace
文本文件是否存在： True
CSV 文件是否存在： True
JSON 文件是否存在： True


### 解释

- `Path('res')` 表示当前目录下的 `res` 文件夹。
- `exists()` 用于判断路径是否存在。
- `mkdir(exist_ok=True)` 用于创建目录；目录已存在时不报错。
- `resolve()` 可以查看路径对应的绝对路径。

### 小练习

1. 打印 `chapter13_text.txt` 的文件名。
2. 打印 `chapter13_text.txt` 的后缀名。
3. 判断 `chapter13_workspace` 是否是目录。


In [3]:
# 1. 文件名
print(text_file.name)

# 2. 后缀名
print(text_file.suffix)

# 3. 判断是否是目录
print(WORK_DIR.is_dir())


chapter13_text.txt
.txt
True


## 2. 读取整个文本文件：`read()`

`read()` 会一次性把文件内容全部读取成一个字符串，适合读取较小的文本文件。


In [4]:
# encoding='utf-8' 表示使用 UTF-8 编码读取中文内容。
# 如果编码写错，可能出现乱码或 UnicodeDecodeError。
with open(text_file, 'r', encoding='utf-8') as f:
    content = f.read()

print(content)
print('内容类型：', type(content))
print('字符数量：', len(content))


Python 文件读写很常用。
with open 可以自动关闭文件。
读取文本时要注意编码。
pathlib 让路径操作更加直观。

内容类型： <class 'str'>
字符数量： 67


### 解释

- `'r'` 表示读取模式。
- `encoding='utf-8'` 用于指定文本编码。
- `with open(...) as f:` 会在代码块结束后自动关闭文件。
- `read()` 适合小文件；大文件不建议一次性全部读入内存。

### 小练习

1. 读取 `chapter13_text.txt` 的全部内容，并统计有多少个字符。
2. 判断文本中是否包含 `pathlib`。
3. 把读取到的内容转成大写后打印。


In [5]:
with open(text_file, 'r', encoding='utf-8') as f:
    text = f.read()

print('字符数：', len(text))
print('是否包含 pathlib：', 'pathlib' in text)
print(text.upper())


字符数： 67
是否包含 pathlib： True
PYTHON 文件读写很常用。
WITH OPEN 可以自动关闭文件。
读取文本时要注意编码。
PATHLIB 让路径操作更加直观。



## 3. 逐行读取文件

逐行读取适合处理较大的文本文件，因为每次只处理一行，不需要一次性把全部内容放进内存。


In [6]:
with open(text_file, 'r', encoding='utf-8') as f:
    for line_number, line in enumerate(f, start=1):
        # 文件中的每一行通常自带换行符 \n
        # strip() 可以去掉首尾空白字符，包括换行符
        clean_line = line.strip()
        print(line_number, clean_line)


with open(text_file, 'r', encoding='utf-8') as f:
    first_line = f.readline()
    other_lines = f.readlines()

print('第一行：', first_line.strip())
print('剩余行列表：', other_lines)


1 Python 文件读写很常用。
2 with open 可以自动关闭文件。
3 读取文本时要注意编码。
4 pathlib 让路径操作更加直观。
第一行： Python 文件读写很常用。
剩余行列表： ['with open 可以自动关闭文件。\n', '读取文本时要注意编码。\n', 'pathlib 让路径操作更加直观。\n']


### 解释

- 直接遍历文件对象 `for line in f` 是最常见的逐行读取方式。
- `readline()` 每次读取一行。
- `readlines()` 会把所有行读取成列表，适合小文件。
- `strip()` 常用于清理行尾换行符。

### 小练习

1. 逐行读取文本文件，只打印包含 `文件` 的行。
2. 统计文本文件一共有多少行。
3. 把每行内容前面加上行号后保存到列表中。


In [7]:
matched_lines = []
numbered_lines = []
line_count = 0

with open(text_file, 'r', encoding='utf-8') as f:
    for line_number, line in enumerate(f, start=1):
        line_count += 1
        clean_line = line.strip()

        if '文件' in clean_line:
            matched_lines.append(clean_line)

        numbered_lines.append(f'{line_number}. {clean_line}')

print('包含“文件”的行：', matched_lines)
print('总行数：', line_count)
print('带行号的内容：', numbered_lines)


包含“文件”的行： ['Python 文件读写很常用。', 'with open 可以自动关闭文件。']
总行数： 4
带行号的内容： ['1. Python 文件读写很常用。', '2. with open 可以自动关闭文件。', '3. 读取文本时要注意编码。', '4. pathlib 让路径操作更加直观。']


## 4. 写入文件：`write()` 和 `writelines()`

写入文件会把程序中的数据保存到磁盘。写入模式 `'w'` 会覆盖原文件内容，追加模式 `'a'` 会在文件末尾继续写。


In [8]:
output_file = WORK_DIR / 'write_demo.txt'

# 'w' 表示写入模式：文件不存在会创建，文件存在会清空原内容。
with open(output_file, 'w', encoding='utf-8') as f:
    f.write('第一行：学习文件写入\n')
    f.write('第二行：write 每次写入一个字符串\n')

print(output_file.read_text(encoding='utf-8'))


lines = [
    '第三行：writelines 可以写入字符串列表\n',
    '第四行：注意自己补上换行符\n'
]

# 'a' 表示追加模式：不会清空原内容，而是在末尾继续写。
with open(output_file, 'a', encoding='utf-8') as f:
    f.writelines(lines)

print(output_file.read_text(encoding='utf-8'))


第一行：学习文件写入
第二行：write 每次写入一个字符串

第一行：学习文件写入
第二行：write 每次写入一个字符串
第三行：writelines 可以写入字符串列表
第四行：注意自己补上换行符



### 解释

- `'w'` 会覆盖原文件，使用前要确认是否真的要清空旧内容。
- `'a'` 会追加内容，适合写日志、记录历史数据。
- `write()` 参数必须是字符串，不能直接写整数、列表、字典。
- `writelines()` 不会自动添加换行符，需要自己在字符串中写 `\n`。

### 小练习

1. 创建 `scores.txt`，写入三名学生和分数。
2. 使用追加模式再添加一名学生。
3. 读取并打印最终文件内容。


In [9]:
scores_file = WORK_DIR / 'scores.txt'

with open(scores_file, 'w', encoding='utf-8') as f:
    f.write('Tom 86\n')
    f.write('Jerry 92\n')
    f.write('Alice 79\n')

with open(scores_file, 'a', encoding='utf-8') as f:
    f.write('Bob 88\n')

print(scores_file.read_text(encoding='utf-8'))


Tom 86
Jerry 92
Alice 79
Bob 88



## 5. CSV 文件读写

CSV 是常见的表格文本格式，字段之间用逗号分隔。Python 标准库提供了 `csv` 模块处理 CSV 文件。


In [10]:
import csv

with open(csv_file, 'r', encoding='utf-8', newline='') as f:
    reader = csv.DictReader(f)
    students = list(reader)

print(students)

# CSV 读出来的内容默认都是字符串。
for student in students:
    student['age'] = int(student['age'])
    student['score'] = int(student['score'])

print(students)


new_csv_file = WORK_DIR / 'passed_students.csv'

with open(new_csv_file, 'w', encoding='utf-8', newline='') as f:
    fieldnames = ['name', 'score']
    writer = csv.DictWriter(f, fieldnames=fieldnames)

    # 写入表头
    writer.writeheader()

    for student in students:
        if student['score'] >= 80:
            writer.writerow({
                'name': student['name'],
                'score': student['score']
            })

print(new_csv_file.read_text(encoding='utf-8'))


[{'name': 'Tom', 'age': '18', 'score': '86'}, {'name': 'Jerry', 'age': '19', 'score': '92'}, {'name': 'Alice', 'age': '18', 'score': '79'}, {'name': 'Bob', 'age': '20', 'score': '88'}]
[{'name': 'Tom', 'age': 18, 'score': 86}, {'name': 'Jerry', 'age': 19, 'score': 92}, {'name': 'Alice', 'age': 18, 'score': 79}, {'name': 'Bob', 'age': 20, 'score': 88}]
name,score
Tom,86
Jerry,92
Bob,88



### 解释

- `csv.DictReader` 会把每一行读取成字典，键来自表头。
- CSV 中读取出的数字默认是字符串，需要手动 `int()` 转换。
- 写 CSV 时建议加 `newline=''`，避免在 Windows 中出现多余空行。
- `writer.writeheader()` 用于写入表头。

### 小练习

1. 读取 `chapter13_students.csv`，计算平均分。
2. 找出最高分学生。
3. 把分数大于等于 85 的学生写入 `excellent_students.csv`。


In [11]:
import csv

with open(csv_file, 'r', encoding='utf-8', newline='') as f:
    reader = csv.DictReader(f)
    students = []
    for row in reader:
        row['age'] = int(row['age'])
        row['score'] = int(row['score'])
        students.append(row)

average_score = sum(student['score'] for student in students) / len(students)
best_student = max(students, key=lambda item: item['score'])

excellent_file = WORK_DIR / 'excellent_students.csv'
with open(excellent_file, 'w', encoding='utf-8', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['name', 'score'])
    writer.writeheader()
    for student in students:
        if student['score'] >= 85:
            writer.writerow({'name': student['name'], 'score': student['score']})

print('平均分：', average_score)
print('最高分学生：', best_student)
print(excellent_file.read_text(encoding='utf-8'))


平均分： 86.25
最高分学生： {'name': 'Jerry', 'age': 19, 'score': 92}
name,score
Tom,86
Jerry,92
Bob,88



## 6. JSON 文件读写

JSON 常用于保存配置、接口数据、结构化信息。Python 使用 `json` 模块处理 JSON。


In [12]:
import json

with open(json_file, 'r', encoding='utf-8') as f:
    config = json.load(f)

print(config)
print('项目名：', config['project'])
print('功能列表：', config['features'])


config['debug'] = False
config['author'] = 'Python learner'

new_json_file = WORK_DIR / 'updated_config.json'

with open(new_json_file, 'w', encoding='utf-8') as f:
    # ensure_ascii=False 可以让中文正常保存，不被转义成 \uXXXX
    # indent=2 可以让 JSON 文件更容易阅读
    json.dump(config, f, ensure_ascii=False, indent=2)

print(new_json_file.read_text(encoding='utf-8'))


{'project': 'python_base', 'debug': True, 'version': '1.0', 'features': ['file', 'os', 'pathlib']}
项目名： python_base
功能列表： ['file', 'os', 'pathlib']
{
  "project": "python_base",
  "debug": false,
  "version": "1.0",
  "features": [
    "file",
    "os",
    "pathlib"
  ],
  "author": "Python learner"
}


### 解释

- `json.load(f)` 从文件读取 JSON，并转换成 Python 字典或列表。
- `json.dump(data, f)` 把 Python 数据写入 JSON 文件。
- `json.loads(text)` 处理 JSON 字符串。
- `json.dumps(data)` 把 Python 数据转换成 JSON 字符串。

### 小练习

1. 读取配置文件，打印 `version`。
2. 给配置新增字段 `level = beginner`。
3. 保存到 `practice_config.json`。


In [13]:
import json

with open(json_file, 'r', encoding='utf-8') as f:
    config = json.load(f)

print('版本：', config['version'])

config['level'] = 'beginner'

practice_json_file = WORK_DIR / 'practice_config.json'
with open(practice_json_file, 'w', encoding='utf-8') as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

print(practice_json_file.read_text(encoding='utf-8'))


版本： 1.0
{
  "project": "python_base",
  "debug": true,
  "version": "1.0",
  "features": [
    "file",
    "os",
    "pathlib"
  ],
  "level": "beginner"
}


## 7. `os` 和 `pathlib` 常用路径操作

`os` 是传统文件系统操作模块，`pathlib` 是更现代的面向对象路径工具。新代码中可以优先使用 `pathlib`。


In [14]:
import os
from pathlib import Path

path = text_file

print('文件名：', path.name)
print('父目录：', path.parent)
print('后缀名：', path.suffix)
print('是否存在：', path.exists())
print('是否文件：', path.is_file())
print('文件大小：', path.stat().st_size, '字节')

# os 模块也可以完成类似操作
print(os.path.basename(path))
print(os.path.dirname(path))
print(os.path.splitext(path))
print(os.path.exists(path))


文件名： chapter13_text.txt
父目录： res
后缀名： .txt
是否存在： True
是否文件： True
文件大小： 147 字节
chapter13_text.txt
res
('res\\chapter13_text', '.txt')
True


### 解释

- `path.name` 获取文件名。
- `path.parent` 获取父目录。
- `path.suffix` 获取扩展名。
- `path.stat().st_size` 获取文件大小。
- `os.path` 适合维护老代码，`pathlib` 更适合新代码。

### 小练习

1. 打印 CSV 文件的绝对路径。
2. 判断 JSON 文件是不是普通文件。
3. 打印 JSON 文件大小。


In [15]:
print(csv_file.resolve())
print(json_file.is_file())
print(json_file.stat().st_size)


C:\Users\11435\Desktop\python_base\python_learn\res\chapter13_students.csv
True
115


## 8. `os` 常用方法速查表

`os` 模块主要用于和操作系统交互，常见用途包括：获取当前目录、拼接路径、判断文件是否存在、创建目录、遍历目录、删除或重命名文件、读取环境变量等。

| 分类 | 方法 | 常见写法 | 作用 | 注意点 |
| --- | --- | --- | --- | --- |
| 当前目录 | `os.getcwd()` | `os.getcwd()` | 获取当前工作目录 | 相对路径会从当前工作目录开始查找 |
| 切换目录 | `os.chdir(path)` | `os.chdir('data')` | 修改当前工作目录 | 会影响后续相对路径，建议用完后切回原目录 |
| 列出目录 | `os.listdir(path)` | `os.listdir('.')` | 返回目录下的文件名和文件夹名列表 | 只返回名称，不包含完整路径 |
| 高效遍历 | `os.scandir(path)` | `for entry in os.scandir(path): ...` | 遍历目录项，可直接判断文件或目录 | 适合需要 `is_file()`、`is_dir()` 的场景 |
| 创建目录 | `os.mkdir(path)` | `os.mkdir('logs')` | 创建单层目录 | 父目录不存在会报错；目录已存在也会报错 |
| 创建多级目录 | `os.makedirs(path, exist_ok=True)` | `os.makedirs('a/b', exist_ok=True)` | 递归创建多级目录 | `exist_ok=True` 表示目录已存在时不报错 |
| 删除空目录 | `os.rmdir(path)` | `os.rmdir('logs')` | 删除空文件夹 | 目录非空会报错 |
| 删除文件 | `os.remove(path)` | `os.remove('a.txt')` | 删除普通文件 | 删除前建议先判断路径是否存在 |
| 重命名/移动 | `os.rename(src, dst)` | `os.rename('a.txt', 'b.txt')` | 重命名或移动文件/目录 | 目标已存在时不同系统表现可能不同 |
| 替换文件 | `os.replace(src, dst)` | `os.replace('new.txt', 'old.txt')` | 用源文件替换目标文件 | 如果目标文件存在，会被覆盖 |
| 文件状态 | `os.stat(path)` | `os.stat('a.txt').st_size` | 获取大小、修改时间等信息 | 返回的是 stat 结果对象 |
| 递归遍历 | `os.walk(path)` | `for root, dirs, files in os.walk(path): ...` | 递归遍历目录树 | 适合批量查找文件 |
| 拼接路径 | `os.path.join(a, b)` | `os.path.join('res', 'a.txt')` | 按当前系统规则拼接路径 | 不要手写 `/` 或 `\\` 拼接路径 |
| 绝对路径 | `os.path.abspath(path)` | `os.path.abspath('a.txt')` | 获取绝对路径 | 文件不存在也能计算绝对路径 |
| 父目录 | `os.path.dirname(path)` | `os.path.dirname('/a/b.txt')` | 获取路径中的目录部分 | 只做字符串路径分析，不检查是否存在 |
| 文件名 | `os.path.basename(path)` | `os.path.basename('/a/b.txt')` | 获取路径末尾文件名 | 路径末尾是目录分隔符时结果可能为空 |
| 拆分扩展名 | `os.path.splitext(path)` | `os.path.splitext('a.txt')` | 拆成文件主名和后缀 | 返回二元组，例如 `('a', '.txt')` |
| 判断存在 | `os.path.exists(path)` | `os.path.exists('a.txt')` | 判断路径是否存在 | 文件或目录存在都返回 `True` |
| 判断文件 | `os.path.isfile(path)` | `os.path.isfile('a.txt')` | 判断是否是普通文件 | 路径不存在返回 `False` |
| 判断目录 | `os.path.isdir(path)` | `os.path.isdir('res')` | 判断是否是目录 | 路径不存在返回 `False` |
| 文件大小 | `os.path.getsize(path)` | `os.path.getsize('a.txt')` | 获取文件大小，单位字节 | 路径不存在会报错 |
| 环境变量 | `os.getenv(name, default)` | `os.getenv('PATH')` | 读取环境变量 | 读不到时返回 `None` 或默认值 |
| 环境变量字典 | `os.environ` | `os.environ['MODE'] = 'dev'` | 读取或设置当前进程环境变量 | 当前 Python 进程内有效，不等于永久修改系统变量 |

下面的代码把这些高频用法串在一起，注释里说明了每一步的使用方式和注意点。


In [16]:
import os
from pathlib import Path

# 这一段代码只在本章的临时演示目录中创建、重命名、删除文件。
# 不会操作系统中的其他目录，真实项目里做删除或覆盖前一定要确认路径。


# 1. 准备演示目录 ------------------------------------------------------------

# Path 只是用来更方便地定位本章 res 目录；本节重点演示 os 的方法。
RES_DIR = Path('res')
if not RES_DIR.exists() and Path('python_learn/res').exists():
    RES_DIR = Path('python_learn/res')

WORK_DIR = RES_DIR / 'chapter13_workspace'
WORK_DIR.mkdir(exist_ok=True)

# os.path.join 用来拼接路径。
# 好处：Windows 会使用反斜杠，macOS/Linux 会使用斜杠，不需要自己判断系统。
os_demo_dir = os.path.join(str(WORK_DIR), 'os_method_demo')

# os.makedirs 可以创建多级目录。
# exist_ok=True 表示目录已经存在时不报错，适合反复运行演示代码。
os.makedirs(os_demo_dir, exist_ok=True)

print('演示目录：', os_demo_dir)


# 2. 当前目录相关 ------------------------------------------------------------

# os.getcwd 获取当前工作目录。
# 注意：相对路径都是相对于当前工作目录来计算的。
old_cwd = os.getcwd()
print('当前工作目录：', old_cwd)

try:
    # os.chdir 可以切换当前工作目录。
    # 这个操作会影响后续所有相对路径，所以通常要谨慎使用。
    os.chdir(os_demo_dir)
    print('切换后的工作目录：', os.getcwd())
finally:
    # 用 finally 确保无论中间是否出错，都切回原来的工作目录。
    os.chdir(old_cwd)

print('恢复后的工作目录：', os.getcwd())


# 3. 路径分析相关 ------------------------------------------------------------

file_path = os.path.join(os_demo_dir, 'note.txt')

# os.path.abspath 获取绝对路径。
# 即使文件还不存在，也能根据当前工作目录计算出绝对路径。
print('绝对路径：', os.path.abspath(file_path))

# os.path.dirname 获取父目录部分。
print('父目录：', os.path.dirname(file_path))

# os.path.basename 获取路径最后一段，也就是文件名。
print('文件名：', os.path.basename(file_path))

# os.path.splitext 把路径拆成“去掉后缀的部分”和“后缀名”。
name_part, suffix_part = os.path.splitext(file_path)
print('主路径：', name_part)
print('后缀：', suffix_part)


# 4. 创建文件、判断文件状态 --------------------------------------------------

# open + write 创建一个普通文本文件。
# os 模块负责路径和文件系统操作，具体读写内容仍然用 open 更常见。
with open(file_path, 'w', encoding='utf-8') as f:
    f.write('第一行：os 常用方法演示\n')
    f.write('第二行：文件状态判断\n')

# os.path.exists 判断路径是否存在；文件或目录存在都会返回 True。
print('路径是否存在：', os.path.exists(file_path))

# os.path.isfile 判断是不是普通文件。
print('是否是文件：', os.path.isfile(file_path))

# os.path.isdir 判断是不是目录。
print('是否是目录：', os.path.isdir(file_path))
print('演示目录是否是目录：', os.path.isdir(os_demo_dir))

# os.path.getsize 获取文件大小，单位是字节。
# 如果路径不存在，会抛出 FileNotFoundError。
print('文件大小：', os.path.getsize(file_path), '字节')

# os.stat 可以获取更完整的文件状态信息。
# st_size 是大小，st_mtime 是最后修改时间戳。
stat_result = os.stat(file_path)
print('stat 文件大小：', stat_result.st_size)
print('stat 修改时间戳：', stat_result.st_mtime)


# 5. 列出目录内容 ------------------------------------------------------------

# 再创建一个子目录和一个子文件，用来演示目录遍历。
sub_dir = os.path.join(os_demo_dir, 'sub')
os.makedirs(sub_dir, exist_ok=True)

sub_file = os.path.join(sub_dir, 'child.txt')
with open(sub_file, 'w', encoding='utf-8') as f:
    f.write('子目录中的文件\n')

# os.listdir 返回指定目录下的名称列表。
# 注意：结果只有名称，不是完整路径。
print('listdir 结果：', os.listdir(os_demo_dir))

# os.scandir 返回目录项对象，可以直接判断它是文件还是目录。
# 处理大量目录项时，os.scandir 通常比 listdir + isfile 更高效。
for entry in os.scandir(os_demo_dir):
    if entry.is_file():
        print('scandir 文件：', entry.name)
    elif entry.is_dir():
        print('scandir 目录：', entry.name)

# os.walk 会递归遍历目录树。
# root 是当前遍历到的目录，dirs 是当前目录下的子目录列表，files 是当前目录下的文件列表。
for root, dirs, files in os.walk(os_demo_dir):
    print('walk 当前目录：', root)
    print('walk 子目录：', dirs)
    print('walk 文件：', files)


# 6. 重命名、替换和删除 ------------------------------------------------------

renamed_path = os.path.join(os_demo_dir, 'renamed_note.txt')

# os.rename 可以重命名文件，也可以把文件移动到其他目录。
# 这里把 note.txt 重命名为 renamed_note.txt。
os.rename(file_path, renamed_path)
print('重命名后旧文件是否存在：', os.path.exists(file_path))
print('重命名后新文件是否存在：', os.path.exists(renamed_path))

replacement_path = os.path.join(os_demo_dir, 'replacement.txt')
with open(replacement_path, 'w', encoding='utf-8') as f:
    f.write('这份内容会替换 renamed_note.txt\n')

# os.replace 会用 replacement_path 替换 renamed_path。
# 如果 renamed_path 已经存在，它会被覆盖。
os.replace(replacement_path, renamed_path)

with open(renamed_path, 'r', encoding='utf-8') as f:
    print('替换后的内容：', f.read().strip())

# os.remove 删除普通文件。
# 删除前先判断存在，可以避免 FileNotFoundError。
if os.path.exists(renamed_path):
    os.remove(renamed_path)

if os.path.exists(sub_file):
    os.remove(sub_file)

print('删除后文件是否存在：', os.path.exists(renamed_path))

# os.rmdir 只能删除空目录。
# 因为 sub_file 已经被删除，所以 sub_dir 现在是空目录，可以删除。
if os.path.isdir(sub_dir):
    os.rmdir(sub_dir)

# 最后删除演示目录。它现在也是空目录。
if os.path.isdir(os_demo_dir):
    os.rmdir(os_demo_dir)

print('演示目录是否还存在：', os.path.exists(os_demo_dir))


# 7. 环境变量 ---------------------------------------------------------------

# os.getenv 用来读取环境变量。
# 第二个参数是默认值：如果环境变量不存在，就返回这个默认值。
python_env = os.getenv('PYTHON_ENV', '未设置')
print('PYTHON_ENV：', python_env)

# os.environ 是环境变量字典。
# 这里设置的变量只影响当前 Python 进程和它创建的子进程，不会永久写入系统环境变量。
os.environ['PYTHON_ENV'] = 'learning'
print('设置后的 PYTHON_ENV：', os.getenv('PYTHON_ENV'))


演示目录： res\chapter13_workspace\os_method_demo
当前工作目录： C:\Users\11435\Desktop\python_base\python_learn
切换后的工作目录： C:\Users\11435\Desktop\python_base\python_learn\res\chapter13_workspace\os_method_demo
恢复后的工作目录： C:\Users\11435\Desktop\python_base\python_learn
绝对路径： C:\Users\11435\Desktop\python_base\python_learn\res\chapter13_workspace\os_method_demo\note.txt
父目录： res\chapter13_workspace\os_method_demo
文件名： note.txt
主路径： res\chapter13_workspace\os_method_demo\note
后缀： .txt
路径是否存在： True
是否是文件： True
是否是目录： False
演示目录是否是目录： True
文件大小： 67 字节
stat 文件大小： 67
stat 修改时间戳： 1781278742.1911573
listdir 结果： ['note.txt', 'sub']
scandir 文件： note.txt
scandir 目录： sub
walk 当前目录： res\chapter13_workspace\os_method_demo
walk 子目录： ['sub']
walk 文件： ['note.txt']
walk 当前目录： res\chapter13_workspace\os_method_demo\sub
walk 子目录： []
walk 文件： ['child.txt']
重命名后旧文件是否存在： False
重命名后新文件是否存在： True
替换后的内容： 这份内容会替换 renamed_note.txt
删除后文件是否存在： False
演示目录是否还存在： False
PYTHON_ENV： 未设置
设置后的 PYTHON_ENV： learning


### 小练习

1. 使用 `os.path.join()` 拼接一个文件路径，例如 `res/demo.txt`。
2. 使用 `os.path.exists()` 判断这个路径是否存在。
3. 使用 `os.makedirs()` 创建一个多级目录，例如 `chapter13_workspace/a/b`。
4. 使用 `os.listdir()` 打印 `chapter13_workspace` 下的内容。
5. 创建一个临时文件，使用 `os.rename()` 重命名，再使用 `os.remove()` 删除。


In [17]:
import os
from pathlib import Path

RES_DIR = Path('res')
if not RES_DIR.exists() and Path('python_learn/res').exists():
    RES_DIR = Path('python_learn/res')

WORK_DIR = RES_DIR / 'chapter13_workspace'
WORK_DIR.mkdir(exist_ok=True)

# 1. 拼接文件路径
practice_file = os.path.join(str(WORK_DIR), 'practice_os.txt')
print('拼接后的路径：', practice_file)

# 2. 判断路径是否存在
print('创建前是否存在：', os.path.exists(practice_file))

# 3. 创建多级目录
practice_dir = os.path.join(str(WORK_DIR), 'a', 'b')
os.makedirs(practice_dir, exist_ok=True)
print('多级目录是否存在：', os.path.isdir(practice_dir))

# 4. 打印目录内容
print('WORK_DIR 内容：', os.listdir(WORK_DIR))

# 5. 创建、重命名、删除临时文件
with open(practice_file, 'w', encoding='utf-8') as f:
    f.write('os 练习文件\n')

renamed_file = os.path.join(str(WORK_DIR), 'practice_os_renamed.txt')
os.rename(practice_file, renamed_file)
print('重命名后是否存在：', os.path.exists(renamed_file))

os.remove(renamed_file)
print('删除后是否存在：', os.path.exists(renamed_file))

# 清理练习中创建的空目录。
# rmdir 只能删除空目录，所以要从最里面一层开始删。
os.rmdir(practice_dir)
os.rmdir(os.path.join(str(WORK_DIR), 'a'))


拼接后的路径： res\chapter13_workspace\practice_os.txt
创建前是否存在： False
多级目录是否存在： True
WORK_DIR 内容： ['a', 'excellent_students.csv', 'logs', 'nested', 'passed_students.csv', 'practice_config.json', 'scores.txt', 'updated_config.json', 'write_demo.txt']
重命名后是否存在： True
删除后是否存在： False


## 9. 创建目录、遍历目录

程序经常需要批量查看目录中的文件，例如统计资源文件、查找日志、筛选指定后缀文件。


In [18]:
demo_dir = WORK_DIR / 'nested'
images_dir = demo_dir / 'images'
texts_dir = demo_dir / 'texts'

# parents=True 表示父目录不存在时一起创建。
images_dir.mkdir(parents=True, exist_ok=True)
texts_dir.mkdir(parents=True, exist_ok=True)

(images_dir / 'a.png').write_text('fake image data', encoding='utf-8')
(texts_dir / 'a.txt').write_text('hello', encoding='utf-8')
(texts_dir / 'b.txt').write_text('python', encoding='utf-8')

print('当前工作目录下的内容：')
for item in WORK_DIR.iterdir():
    print(item.name, '目录' if item.is_dir() else '文件')

print('递归查找所有 txt 文件：')
for path in WORK_DIR.rglob('*.txt'):
    print(path)


当前工作目录下的内容：
excellent_students.csv 文件
logs 目录
nested 目录
passed_students.csv 文件
practice_config.json 文件
scores.txt 文件
updated_config.json 文件
write_demo.txt 文件
递归查找所有 txt 文件：
res\chapter13_workspace\scores.txt
res\chapter13_workspace\write_demo.txt
res\chapter13_workspace\nested\texts\a.txt
res\chapter13_workspace\nested\texts\b.txt


### 解释

- `iterdir()` 只遍历当前目录的第一层内容。
- `rglob('*.txt')` 会递归查找所有 `.txt` 文件。
- `mkdir(parents=True)` 可以一次性创建多级目录。
- 用 `is_dir()` 和 `is_file()` 可以区分目录和文件。

### 小练习

1. 统计 `WORK_DIR` 下有多少个 `.txt` 文件。
2. 打印所有文件的文件名，不包含目录名。
3. 创建一个 `logs` 目录，并在里面写入 `app.log`。


In [19]:
txt_files = list(WORK_DIR.rglob('*.txt'))
print('txt 文件数量：', len(txt_files))

for path in WORK_DIR.rglob('*'):
    if path.is_file():
        print(path.name)

logs_dir = WORK_DIR / 'logs'
logs_dir.mkdir(exist_ok=True)
(logs_dir / 'app.log').write_text('程序启动成功\n', encoding='utf-8')

print((logs_dir / 'app.log').read_text(encoding='utf-8'))


txt 文件数量： 4
excellent_students.csv
passed_students.csv
practice_config.json
scores.txt
updated_config.json
write_demo.txt
app.log
a.png
a.txt
b.txt
程序启动成功



## 10. 重命名、移动和删除文件

重命名和删除属于会改变文件系统状态的操作，真实项目中要格外谨慎。学习时建议只操作自己创建的临时文件。


In [20]:
temp_file = WORK_DIR / 'temp_name.txt'
renamed_file = WORK_DIR / 'renamed_name.txt'

temp_file.write_text('这是一个临时文件', encoding='utf-8')
print('创建后：', temp_file.exists())

# rename 可以重命名，也可以移动文件。
temp_file.rename(renamed_file)
print('旧文件是否存在：', temp_file.exists())
print('新文件是否存在：', renamed_file.exists())

# unlink 用于删除文件。
renamed_file.unlink()
print('删除后新文件是否存在：', renamed_file.exists())


创建后： True
旧文件是否存在： False
新文件是否存在： True
删除后新文件是否存在： False


### 解释

- `rename()` 可以重命名文件，也可以把文件移动到新路径。
- `unlink()` 用于删除文件。
- 删除前建议先用 `exists()` 判断文件是否存在。
- 不要对不熟悉的目录做批量删除操作。

### 小练习

1. 创建 `old.txt`，内容随意。
2. 把它重命名为 `new.txt`。
3. 读取 `new.txt` 后删除它。


In [21]:
old_file = WORK_DIR / 'old.txt'
new_file = WORK_DIR / 'new.txt'

old_file.write_text('准备重命名', encoding='utf-8')
old_file.rename(new_file)

print(new_file.read_text(encoding='utf-8'))

if new_file.exists():
    new_file.unlink()

print('new.txt 是否还存在：', new_file.exists())


准备重命名
new.txt 是否还存在： False


## 11. 常见错误总结

1. 路径写错，导致 `FileNotFoundError`。
2. 读取中文文件时忘记写 `encoding='utf-8'`，导致乱码或解码错误。
3. 使用 `'w'` 模式误覆盖原文件。
4. 忘记关闭文件；建议统一使用 `with open(...)`。
5. 把数字、列表、字典直接传给 `write()`；`write()` 只能写字符串。
6. CSV 读取出的数字其实是字符串，计算前要转换类型。
7. JSON 中的键必须使用双引号，不能写成 Python 字典那样的单引号。
8. 删除文件前没有确认路径，误删重要文件。
9. 在不同运行目录下使用相对路径，导致同一段代码有时能跑、有时报错。
10. 混用 `\` 路径分隔符时忘记转义；使用 `Path` 可以减少这类问题。
